In [1]:
# ============================================================
# 1. Imports and experiment settings
# ============================================================

from pathlib import Path
import ast
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# This is the directory structure used by the original notebook.
# Change this path if the result files are stored elsewhere.
import os


MAIN_DIR = Path('../../results/random_configurations')

DATASETS = ['D2', 'D3', 'D4', 'D8', 'D9', 'D10']
THRESHOLDS = [85, 90, 95]

METHOD_FILES = {
    'LSH': Path('lsh_blocker') / '{dataset}' / 'lsh_blocker.csv',
    'LSH+BP': Path('filtering') / '{dataset}' / 'block_purging.csv',
    'LSH+BF': Path('filtering') / '{dataset}' / 'block_filtering.csv',
    'LSH+BP+BF': Path('filtering') / '{dataset}' / 'combination.csv',
}

METHOD_ORDER = ['LSH', 'LSH+BP', 'LSH+BF', 'LSH+BP+BF']

In [2]:
# ============================================================
# 2. Load all experiment results
# ============================================================

def get_method_file(method, dataset):
    """Return the CSV path for a method and dataset."""
    return MAIN_DIR / METHOD_FILES[method].as_posix().format(dataset=dataset)


def load_results():
    """Load all result CSVs into a dictionary keyed by dataset and method."""
    data = {}
    missing = []

    for dataset in DATASETS:
        data[dataset] = {}

        for method in METHOD_ORDER:
            path = get_method_file(method, dataset)

            if not path.exists():
                missing.append(str(path))
                continue

            df = pd.read_csv(path)

            # Make sure the metrics used for selection are numeric.
            for col in ['Precision %', 'Recall %', 'F1 %', 'runtime',
                    'candidate_pairs', 'total_runtime', 'Unique Precision %', 'Unique F1 %', 'unique_candidate_pairs']:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors='coerce')

            data[dataset][method] = df

    if missing:
        print('WARNING: The following result files were not found:')
        for path in missing:
            print('  ', path)

    return data


results = load_results()

print('Loaded result files:')
for dataset in DATASETS:
    for method in METHOD_ORDER:
        if method in results.get(dataset, {}):
            print(f'  {dataset:>3} | {method:<9} | {len(results[dataset][method]):,} configurations')

Loaded result files:
   D2 | LSH       | 100 configurations
   D2 | LSH+BP    | 100 configurations
   D2 | LSH+BF    | 100 configurations
   D2 | LSH+BP+BF | 100 configurations
   D3 | LSH       | 100 configurations
   D3 | LSH+BP    | 100 configurations
   D3 | LSH+BF    | 100 configurations
   D3 | LSH+BP+BF | 100 configurations
   D4 | LSH       | 100 configurations
   D4 | LSH+BP    | 100 configurations
   D4 | LSH+BF    | 100 configurations
   D4 | LSH+BP+BF | 100 configurations
   D8 | LSH       | 100 configurations
   D8 | LSH+BP    | 100 configurations
   D8 | LSH+BF    | 100 configurations
   D8 | LSH+BP+BF | 100 configurations
   D9 | LSH       | 100 configurations
   D9 | LSH+BP    | 100 configurations
   D9 | LSH+BF    | 100 configurations
   D9 | LSH+BP+BF | 100 configurations
  D10 | LSH       | 100 configurations
  D10 | LSH+BP    | 100 configurations
  D10 | LSH+BF    | 100 configurations
  D10 | LSH+BP+BF | 100 configurations


In [3]:
# ============================================================
# 3. Configuration-selection rule
# ============================================================

def select_configuration(df, threshold):
    """
    Select exactly one configuration according to the requested rule.

    Rule:
      1. Keep configurations with Recall % >= threshold.
      2. If any exist, choose highest precision, then highest recall.
      3. Otherwise choose highest recall, then highest precision,
         and mark the result as a fallback.
    """

    required = ['Precision %', 'Recall %']
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise ValueError(f'Missing required columns: {missing_cols}')

    candidates = df.dropna(subset=required).copy()

    if candidates.empty:
        return None

    # First choice: configurations that meet the requested recall.
    eligible = candidates[candidates['Recall %'] >= threshold]

    if not eligible.empty:
        selected = (
            eligible
            .sort_values(
                by=['Precision %', 'Recall %'],
                ascending=[False, False],
                kind='mergesort'
            )
            .iloc[0]
        )
        selection_type = 'Threshold met'
        fallback = False
    else:
        # No configuration reaches the target: maximize recall,
        # using precision as the secondary criterion.
        selected = (
            candidates
            .sort_values(
                by=['Recall %', 'Precision %'],
                ascending=[False, False],
                kind='mergesort'
            )
            .iloc[0]
        )
        selection_type = 'Fallback: max recall'
        fallback = True

    return {
        'row': selected,
        'selection_type': selection_type,
        'fallback': fallback,
    }

In [7]:
# ============================================================
# 4. Build the selected-results dataframe
# ============================================================

def build_selected_results(results):
    rows = []

    for dataset in DATASETS:
        for threshold in THRESHOLDS:
            for method in METHOD_ORDER:

                df = results.get(dataset, {}).get(method)

                if df is None:
                    rows.append({
                        'Dataset': dataset,
                        'Target Recall': threshold,
                        'Method': method,
                        'Selection': 'No result file',
                        'Fallback': True,
                        'Original Row Index': np.nan, # <--- Added
                        'Precision %': np.nan,
                        'Recall %': np.nan,
                        'Candidate pairs': np.nan,
                        'Runtime': np.nan,
                        'F1 %': np.nan,
                        'Configuration': None,
                        'Ratio': np.nan,
                        'Total Runtime': np.nan,
                        'Unique Precision %': np.nan,
                        'Unique F1 %': np.nan,
                        'Unique Candidate Pairs': np.nan
                    })
                    continue

                selection = select_configuration(df, threshold)

                if selection is None:
                    rows.append({
                        'Dataset': dataset,
                        'Target Recall': threshold,
                        'Method': method,
                        'Selection': 'No valid configuration',
                        'Fallback': True,
                        'Original Row Index': np.nan, # <--- Added
                        'Precision %': np.nan,
                        'Recall %': np.nan,
                        'Candidate pairs': np.nan,
                        'Runtime': np.nan,
                        'F1 %': np.nan,
                        'Configuration': None,
                        'Ratio': np.nan,
                        'Total Runtime': np.nan,
                        'Unique Precision %': np.nan,
                        'Unique F1 %': np.nan,
                        'Unique Candidate Pairs': np.nan
                    })
                    continue

                row = selection['row']

                rows.append({
                    'Dataset': dataset,
                    'Target Recall': threshold,
                    'Method': method,
                    'Selection': selection['selection_type'],
                    'Fallback': selection['fallback'],
                    'Original Row Index': row.name, # <--- GRABS ORIGINAL INDEX HERE
                    'Precision %': row.get('Precision %', np.nan),
                    'Recall %': row.get('Recall %', np.nan),
                    'Candidate pairs': row.get('candidate_pairs', np.nan),
                    'Runtime': row.get('runtime', np.nan),
                    'F1 %': row.get('F1 %', np.nan),
                    'Configuration': row.get('config', None),
                    'Ratio': row.get('ratio', np.nan),
                    'Total Runtime': row.get('total_runtime', row.get('runtime', np.nan)),
                    'Unique Precision %': row.get('Unique Precision %', np.nan),
                    'Unique F1 %': row.get('Unique F1 %', np.nan),
                    'Unique Candidate Pairs': row.get('unique_candidate_pairs', np.nan)
                })


    out = pd.DataFrame(rows)

    out['Method'] = pd.Categorical(
        out['Method'], categories=METHOD_ORDER, ordered=True
    )
    out['Dataset'] = pd.Categorical(
        out['Dataset'], categories=DATASETS, ordered=True
    )

    return out.sort_values(
        ['Target Recall', 'Dataset', 'Method']
    ).reset_index(drop=True)


selected_results = build_selected_results(results)

display(selected_results[
    ['Dataset', 'Target Recall', 'Method', 'Selection', 'Original Row Index',
     'Precision %', 'Unique Precision %', 'Recall %', 'Candidate pairs', 'Unique Candidate Pairs', 'Runtime', 'Total Runtime', ]
].head(12))


,Dataset,Target Recall,Method,Selection,Original Row Index,Precision %,Unique Precision %,Recall %,Candidate pairs,Unique Candidate Pairs,Runtime,Total Runtime
0,D2,85,LSH,Threshold met,69,0.95,1.71,85.34,95373,53000,0.67,0.67
1,D2,85,LSH+BP,Fallback: max recall,30,4.21,7.57,83.18,21038,11695,0.09,0.76
2,D2,85,LSH+BF,Threshold met,92,1.65,2.74,86.09,55441,33403,0.05,0.86
3,D2,85,LSH+BP+BF,Fallback: max recall,16,10.57,18.87,79.79,8029,4500,0.11,1.00
4,D3,85,LSH,Threshold met,52,0.25,0.36,87.57,381786,265850,1.48,1.48
5,D3,85,LSH+BP,Fallback: max recall,67,2.07,5.67,81.49,43355,15838,0.14,2.07
6,D3,85,LSH+BF,Threshold met,38,0.42,0.58,85.75,225427,163014,0.14,1.70
7,D3,85,LSH+BP+BF,Fallback: max recall,48,3.74,9.15,82.67,24380,9959,0.19,2.62
8,D4,85,LSH,Threshold met,31,0.70,13.26,99.46,313191,16604,3.00,3.00
9,D4,85,LSH+BP,Threshold met,52,14.37,34.03,87.31,13449,5681,0.36,2.29


In [10]:
# ============================================================
# 7. Inspect exactly which configurations were selected
# ============================================================

# Display the complete set of selected configurations as a table.
# This is 6 datasets × 3 thresholds × 4 methods = 72 rows.

# configuration_details = selected_results.reset_index(names='Original Index')
configuration_details = selected_results.copy()

configuration_details_display = configuration_details[
    [
        'Dataset',
        'Target Recall',
        'Method',
        'Selection',
        'Original Row Index',
        'Precision %',
        'Recall %',
        'F1 %',
        'Candidate pairs',
        'Runtime',
        'Ratio',
        'Total Runtime',
        'Unique Precision %',
        'Unique F1 %',
        'Unique Candidate Pairs',
        'Configuration'
    ]
].copy()

display(
    configuration_details_display.style
    .format({
        'Precision %': '{:.2f}',
        'Recall %': '{:.2f}',
        'F1 %': '{:.2f}',
        'Candidate pairs': '{:,.0f}',
        'Runtime': '{:.2f}',
        'Ratio': '{:.2f}',
        'Total Runtime': '{:.2f}',
        'Unique Precision %': '{:.2f}',
        'Unique F1 %': '{:.2f}',
        'Unique Candidate pairs': '{:,.0f}',
    }, na_rep='—')
    .hide(axis='index')
)

configuration_details_display.to_csv('../../scripts/data_manipulation/selected_configurations.csv', index=False)

Dataset,Target Recall,Method,Selection,Original Row Index,Precision %,Recall %,F1 %,Candidate pairs,Runtime,Ratio,Total Runtime,Unique Precision %,Unique F1 %,Unique Candidate Pairs,Configuration
D2,85,LSH,Threshold met,69,0.95,85.34,1.88,"95,373",0.67,—,0.67,1.71,3.36,53000,"{'attributes': ['name'], 'size': 64, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D2,85,LSH+BP,Fallback: max recall,30,4.21,83.18,8.01,"21,038",0.09,—,0.76,7.57,13.87,11695,"{'attributes': ['name'], 'size': 256, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 4}"
D2,85,LSH+BF,Threshold met,92,1.65,86.09,3.24,"55,441",0.05,0.80,0.86,2.74,5.32,33403,"{'attributes': ['name'], 'size': 1024, 'offset': 0, 'num_hashes': 18, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D2,85,LSH+BP+BF,Fallback: max recall,16,10.57,79.79,18.67,"8,029",0.11,0.80,1.00,18.87,30.52,4500,"{'attributes': ['name'], 'size': 512, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,LSH,Threshold met,52,0.25,87.57,0.50,"381,786",1.48,—,1.48,0.36,0.72,265850,"{'attributes': ['title'], 'size': 512, 'offset': 0, 'num_hashes': 9, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,LSH+BP,Fallback: max recall,67,2.07,81.49,4.04,"43,355",0.14,—,2.07,5.67,10.60,15838,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 3, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,LSH+BF,Threshold met,38,0.42,85.75,0.83,"225,427",0.14,0.80,1.70,0.58,1.15,163014,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 15, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 3}"
D3,85,LSH+BP+BF,Fallback: max recall,48,3.74,82.67,7.15,"24,380",0.19,0.80,2.62,9.15,16.47,9959,"{'attributes': ['title'], 'size': 512, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 4}"
D4,85,LSH,Threshold met,31,0.70,99.46,1.40,"313,191",3.00,—,3.00,13.26,23.40,16604,"{'attributes': ['title'], 'size': 1024, 'offset': 0, 'num_hashes': 17, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 4}"
D4,85,LSH+BP,Threshold met,52,14.37,87.31,24.68,"13,449",0.36,—,2.29,34.03,48.97,5681,"{'attributes': ['title'], 'size': 64, 'offset': 0, 'num_hashes': 1, 'hashing_type': 'salted_qgrams', 'salt': '', 'qgrams': 4}"
